In [1]:
import os
os.environ["MKL_NUM_THREADS"] = "1" # Or "4"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

from timeit import default_timer as timer
from joblib import Parallel, delayed

import numpy as np
import scipy.sparse as sp
from scipy.special import eval_chebyu
import scipy
import plotly.express as px

np.random.seed(0)


In [2]:
# -------- FUNCTIONS --------
def Gaussian3D(f, x, y=0, z=0, x0=0, y0=0, z0=0):
    return np.exp(-np.pi**2 * f**2 * ((x - x0)**2 + (y - y0)**2 + (z - z0)**2))

def Ricker3D(f, x, y=0, z=0, x0=0, y0=0, z0=0):
    r2 = (x - x0)**2 + (y - y0)**2 + (z - z0)**2
    return (1 - 2 * np.pi**2 * f**2 * r2) * np.exp(-np.pi**2 * f**2 * r2)

def get_mats(m):
    # Material matrices
    M = sp.diags(m)                          # Mass
    M_sqrt = sp.diags(np.sqrt(m))            # Square root of mass
    M_inv = sp.diags(1/m)                    # Mass inverse
    M_sqrt_inv = sp.diags(1/np.sqrt(m))      # Inverse square root
    
    return (M, M_sqrt, M_inv, M_sqrt_inv)

def _prefix_products(U, T, n_jobs=-1):
    """
    Return [I, U, U², …, U^(T-1)] using the binary-tree (exponents-of-2)
    parallel prefix.  Depth = ⌈log₂ T⌉; multiplications run in parallel.
    """
    G = [sp.eye(U.shape[0], dtype=U.dtype, format='csc')]   # U^0
    P = U.copy()                                   # U^(2^0)
    block = 1                                      # size of last “stripe”

    while len(G) < T:
        # multiply the most recent `block` entries by the current power `P`
        chunk = G[-block:]
        need = min(block, T - len(G))              # don’t overshoot at the end
        new = Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(lambda P, M: P @ M)(P, M) for M in chunk[:need]
        )
        G.extend(new)                              # append U^(k+2^d)
        P = P @ P                                  # square: U^(2·2^d)
        block <<= 1                                # double the stripe width
    return G

def get_A_p(H, dt, T, p_x, p_b, n_jobs=-1):
    # Unitary evolution operator
    #time= timer()
    U = sp.linalg.expm(-1j * H * dt)
    #print(f"Time taken for expm: {timer() - time:.4f} seconds")

    # Green’s functions: [I, U, U², …, U^(T-1)]
    #time = timer()
    G_list = _prefix_products(U, T, n_jobs)
    #print(f"Time taken for Green's functions: {timer() - time:.4f} seconds")

    # Linear system (Projected, Block-Toeplitz)
    A_p = sp.block_array([[G_list[j-i][p_x.astype(bool)[j], :][:, p_b.astype(bool)[i]] if i <= j else None for i in range(T)] for j in range(T)])
    
    # Projector-Weighting
    A_p = np.diag(p_x[p_x.astype(bool)]) @ A_p @ np.diag(p_b[p_b.astype(bool)])

    return A_p

def get_LS(T, dt, m, k, D, p_x, p_b, lam1=0.0, lam2=0.00): # lam2=0.05
    # Material matrices
    (_, M_sqrt, _, M_sqrt_inv) = get_mats(m)                                                # Mass matrices
    (_, K_sqrt, _, K_sqrt_inv) = get_mats(k)                                                # Spring matrices

    # Material Transform
    Lam1 = np.diag([1] * len(m)) * lam1                                                     # Small dissipation for mass
    Lam2 = np.diag([1] * len(k)) * lam2                                                     # Small dissipation for spring
    
    B_sqrt = sp.block_diag([M_sqrt, K_sqrt_inv], format='csc')                              # Material Transform
    B_sqrt_inv = sp.block_diag([M_sqrt_inv, K_sqrt], format='csc')                          # Inverse Material Transform
    AH = sp.block_array([[-D.T @ Lam1 @ D, -D.T], [D, -Lam2]])                                          # Incidence matrix
    H = 1j * (B_sqrt_inv @ AH @ B_sqrt_inv)                                                 # Hamiltonian -> O(n)
    
    # Linear system
    A_p = get_A_p(H, dt, T, p_x, p_b)                                                       # System matrix
    
    # Normalize spectral norm (Unnecessary if not quantum)
    #A_p /= T
    
    return A_p, B_sqrt, B_sqrt_inv


In [3]:
# -------- PARAMETERS --------
# Grid parameters
n = 5
N0 = 2**n                                                       # Number of masses
M0 = 2**n                                                       # Number of springs
N = N0 + M0                                                     # Total number of nodes

# Simulation parameters
dt = 2**n / 20                                                 # Time step
T = 30                                                    # Number of time steps

# True medium
m0 = np.ones(N0)                                                # Masses GT

a0_a = 0                                                      # Amplification factor
a0_b = 2.5                                                      # Amplification factor
g0_a = np.abs(Ricker3D(1.08, np.arange(N0), x0=N0//2))          # Function GT a
g0_b = np.abs(Gaussian3D(0.133, np.arange(N0), x0=N0//3.5))                # Function GT b
# g0_b = np.zeros(N0)
# g0_b[N0//2] = 1

In [4]:
#k0 = 1 + (a0_a+a0_b) * g0_a + a0_b * g0_b                       # Spring constants GT
k0 = 1 + a0_b * g0_b
# k0 = np.ones(N0)
#k0 = np.random.rand(N0)
display(px.line(k0, title='Spring constants'))

# Source distribution
b0 = np.zeros((T, N))       
b0[0, :N0] += Gaussian3D(0.1, np.arange(N0), x0=N0//2)       # Initial position
#b0[0, :N0] += Ricker3D(7/(2**n), np.arange(N0), x0=N0//2.8)         # Initial position
display(px.imshow(b0[:N0], title='Source distribution (pressures)'))

# Data points (all times)
gap = N // 2
p_x = np.zeros((T, N))                                          # Data projection
p_x += np.tile(np.tile(np.hstack([np.ones(1), np.zeros(gap-2), np.ones(1)]), N//gap), (T, 1))
#p_x += np.ones((T, N))
display(px.imshow(p_x, title=f'Data projection (NNZ={np.count_nonzero(p_x)})'))

# Source points (initial time)
p_b = np.zeros((T, N))                                   
#p_b[0] += (b0[0] == b0[0].max()).astype(int) # Source projection
p_b[0] += (np.abs(b0[0])**2 >= 1e-3).astype(int)
#p_b[0] = np.ones(N)   # All sources
#p_b[0] += np.round(b0[0], 4)
#p_b[0] += np.ones(N)
#p_b += np.ones((T, N))  # Observe all components at all times and places
display(px.imshow(p_b, title=f'Source projection (NNZ={np.count_nonzero(p_b)})'))


In [5]:
# -------- GENERAL OPERATORS --------
def fornberg(x, m):
    """
    Calculate the weights of a finite difference approximation of the m-th derivative
    with maximal order of accuracy at 0 using the nodes x.
    
    This function implements the algorithm from Fornberg (1998):
      "Calculation of Weights in Finite Difference Formulas"
      SIAM Rev. 40.3, pp. 685-691.
    
    Parameters
    ----------
    x : array_like
        The nodes at which the approximation is based.
    m : int
        The derivative order.
    
    Returns
    -------
    weights : ndarray
        An array of weights corresponding to the nodes in x for approximating the m-th derivative.
    """
    x = np.sort(np.array(x, dtype=float))
    z = 0.0
    n = len(x) - 1
    c = np.zeros((len(x), m + 1), dtype=float)
    c1 = 1.0
    c4 = x[0] - z
    c[0, 0] = 1.0
    
    for i in range(1, n + 1):
        mn = min(i, m)
        c2 = 1.0
        c5 = c4
        c4 = x[i] - z
        for j in range(i):
            c3 = x[i] - x[j]
            c2 *= c3
            if j == i - 1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1 * (k * c[i-1, k-1] - c5 * c[i-1, k]) / c2
                c[i, 0] = -c1 * c5 * c[i-1, 0] / c2
            for k in range(mn, 0, -1):
                c[j, k] = (c4 * c[j, k] - k * c[j, k-1]) / c3
            c[j, 0] = c4 * c[j, 0] / c3
        c1 = c2
    return c[:, m]

def FD(N, coeffs=[1, -1]):
    """Finite Difference operator for first derivative (Periodic BCs)."""
    fd = np.zeros((N, N))
    l = len(coeffs)
    offsets = np.arange(-l//2, l//2)
    for offset, coeff in zip(offsets, coeffs):
        fd += np.roll(coeff*np.eye(N), offset, axis=1)
    return fd

# Incidence Matrix
D = sp.diags([np.ones(N0), -np.ones(N0-1)], [0, 1], format='csc')                       # Incidence matrix
D[0,0]= 1                                                                               # Boundary condition (left, DBC) (TODO: Last Spring DOF is wrapped around!)
D[-1,-1] = 0                                                                            # Boundary condition (right, DBC)

# Derivative operator D
# order = 8                                                                               # Accuracy order of the derivative
# dx = 1                                                                                  # Grid spacing
# pos = np.arange(-order, order) + 0.5
# coeffs = fornberg(pos, dx)
# D = sp.csr_array(FD(N//2, coeffs)).T



px.imshow(D.toarray())


In [6]:
# -------- GROUND TRUTH --------
# Ground truth matrices
A_p_true, B_sqrt_true, B_sqrt_inv_true = get_LS(T, dt, m0, k0, D, p_x, p_b)
display(px.imshow(A_p_true.real, title='System matrix A (true medium)'))
#display(px.imshow(A_p_true[:, 2].real.reshape(T, -1), title='Impulse response at center source, pressure component'))

# Initial conditions
b0 /= np.linalg.norm(b0)                                                                # Normalize initial conditions
b_true = (B_sqrt_true @ b0.T).T                                                         # Right-hand side GT

# Project the initial conditions (source region)
b_true = b_true.flatten()[p_b.astype(bool).flatten()]

# Solve the forward problem (including the measurement projection)
x_p = A_p_true @ b_true

# Compute data norm
x_norm = np.linalg.norm(x_p)

# Data resolution matrix (only geometric!)
R_true = (A_p_true @ np.linalg.pinv(A_p_true)).real

# Loss computation
E = np.linalg.norm(R_true @ x_p) / x_norm
loss = 1 - E

# Display results
print(f'Expected value: {E:.10f}')

print(f'Geometric loss for true medium, {loss:.10f}\n')

# Show wave field (only if P_b = I)
# display(px.imshow(x_p.reshape(T, N).real, title='Wave field (true medium)'))

# Compute Maximum-likelihood source
b_ml = np.linalg.lstsq(A_p_true, x_p, rcond=None)[0]  # Solve the linear system
b_full = p_b.copy()
b_full[b_full.nonzero()] = b_ml.real
#display(px.imshow(b_full.reshape(T, N).real, title='Maximum-likelihood source'))
#display(px.imshow(b0[:N0], title='True source'))

# Show wave field
x_full = p_x.copy()
x_full[x_full.nonzero()] = x_p.real
#display(px.imshow(x_full, title='Sampled Wave Field').show())


Expected value: 1.0000000000
Geometric loss for true medium, -0.0000000000



In [7]:
# Data resolution matrix
display(px.imshow(R_true.real))                                    # Data resolution matrix
display(px.imshow((np.linalg.pinv(A_p_true) @ A_p_true ).real))    # Source resolution matrix


In [8]:
# Compute upper bound for condition number (Exact if P is identity)
theta = np.pi / (4 * T + 2)
k_max = eval_chebyu((2*T-2), np.cos(theta))
print(f'Condition number upper bound: {k_max:.4f}')

# Condition number of the system
k = np.linalg.cond(A_p_true)
print(f'Condition number of the projected system: {k:.4f}')

# Singular values
_, S, _ = scipy.linalg.svd(A_p_true)
print(f'Singular values: max={np.max(S):.4f}, min={np.min(S):.4f}')
px.line(S)


Condition number upper bound: 38.7866
Condition number of the projected system: 11.6209
Singular values: max=1.5739, min=0.1354


In [9]:
# -------- GROUND TRUTH --------
a_a, a_b = 0, 0
m_guess = np.ones(N0)
k_guess = 1 + np.random.rand(N0) * 0

# Guess effective system
A_p_guess, B_sqrt_guess, B_sqrt_inv_guess = get_LS(T, dt, m_guess, k_guess, D, p_x, p_b)

# Data resolution matrix
R_guess = (A_p_guess @ np.linalg.pinv(A_p_guess)).real

# Loss computation
residual = (np.eye(R_guess.shape[0]) - R_guess) @ x_p / x_norm
print(f'Residual norm: {np.linalg.norm(residual)}')

# Guess effective data system (pseudoinverse backpropagation)
C_p_guess, _, _ = get_LS(T, dt, m_guess, k_guess, D, p_x, np.ones((T, N)))

# Guess Pseudo-inverse
PT = np.kron(np.ones((T, 1)), np.eye(N)) / np.sqrt(T) # Temporal Independence
J_ml = np.linalg.pinv(C_p_guess @ PT)
J_adj = PT.T @ C_p_guess.T

####
# Maximum likelihood image
b_ml_image = (J_ml @ residual)**2

# Adjoint image
b_adjoint_image = (J_adj @ residual)**2

# Display Fields
display(px.line(b_ml_image.real, title='Maximum-likelihood misfit source'))
display(px.line(b_adjoint_image.real, title='Adjoint misfit source'))
display(px.line(k0 - k_guess, title='Spring constant model error'))


Residual norm: 0.573335100041336


In [10]:
# --------- INVERSION --------
# Inversion parameter
a_a_range = np.linspace(0, 5, 15)                                        # Range of amplification factors
a_b_range = np.linspace(0, 5, 15)                                        # Range of spring constants

# Static mass
m = np.ones(N0)                                                             # Masses

# Losses
euclidean_losses = np.zeros((len(a_a_range), len(a_b_range)))
geometric_losses = np.zeros((len(a_a_range), len(a_b_range)))
amplitude_losses = np.zeros((len(a_a_range), len(a_b_range)))

# Singular functions
f_hard = lambda x: x!=0
f_soft = lambda x, lam: x/(x + lam)

# Inversion loop
for i, a_a in enumerate(a_a_range):
    for j, a_b in enumerate(a_b_range):
        # Inversion parameters
        #k = 1 + (a_a+a_b) * g0_a + a_b * g0_b
        k = 1 + a0_b * g0_b + a_a * g0_a
        assert np.all(k > 0)                                          # Check if spring constants are positive

        # System matrices
        A_p, B_sqrt, B_sqrt_inv = get_LS(T, dt, m, k, D, p_x, p_b)

        ########
        # Mat. transformed true source (b0 is normed)
        b = (B_sqrt @ b0.T).T
        b = b.flatten()[p_b.astype(bool).flatten()]
        
        # Forward solve (quantum projector, sends to Null space)
        x = A_p @ b
        
        ########
        # Compute SVD of data-space Gramian
        S, U = scipy.linalg.eigh(A_p @ A_p.conj().T)
        S = S.round(10)
        
        # Define regularization parameter
        lam  = S.max() / 2

        # Hard projector
        R = U @ np.diag(f_hard(S)) @ U.conj().T
        
        # Soft projector
        #R = U @ np.diag(f_soft(S, lam)) @ U.conj().T
        
        # ---- LOSSES ----
        # Euclidean Loss
        euclidean_losses[i,j] = np.linalg.norm(x - x_p)**2
        
        # Geometric Loss (unnormalized, x_norm)
        geometric_losses[i,j] = np.linalg.norm((np.eye(R.shape[0]) - R) @ x_p)**2
        
        # Amplitude Loss (unnormalized, x_norm)
        amplitude_losses[i,j] = np.linalg.norm(x - R @ x_p)**2


        print(f'Loss for a={a_a:.4f}/b={a_b:.4f}, euclidean={euclidean_losses[i,j]:.6f}, geometric={geometric_losses[i,j]:.6f}, amplitude={amplitude_losses[i,j]:.6f}, diff={(euclidean_losses[i,j] - (geometric_losses[i,j] + amplitude_losses[i,j])):.6f}')


Loss for a=0.0000/b=0.0000, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=0.3571, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=0.7143, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=1.0714, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=1.4286, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=1.7857, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=2.1429, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=2.5000, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=2.8571, euclidean=0.000000, geometric=0.000000, amplitude=0.000000, diff=-0.000000
Loss for a=0.0000/b=3.2143, euclidean=0.000000, geometric=0.000000, ampli

In [11]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=4, cols=3, column_titles=['Euclidean Loss', 'Geometric Loss', 'Amplitude Loss'], row_titles=['Loss Field', 'Gradient x', 'Gradient y'], shared_xaxes=True, shared_yaxes=True)

# Define shared color scale limits for each row
cmin_row1, cmax_row1 = 0, np.max(euclidean_losses)

euclidean_fft = np.fft.fftshift(np.fft.fft2(euclidean_losses, norm='ortho'))
geometric_fft = np.fft.fftshift(np.fft.fft2(geometric_losses, norm='ortho'))
amplitude_fft = np.fft.fftshift(np.fft.fft2(amplitude_losses, norm='ortho'))

cmin_row2, cmax_row2 = -np.max(np.abs(np.gradient(euclidean_losses)[0])), np.max(np.abs(np.gradient(euclidean_losses)[0]))
cmin_row3, cmax_row3 = -np.max(np.abs(np.gradient(euclidean_losses)[1])), np.max(np.abs(np.gradient(euclidean_losses)[1]))
cmin_row4, cmax_row4 = 0, 0.001

fig.add_trace(go.Heatmap(z=euclidean_losses, zmin=cmin_row1, zmax=cmax_row1, coloraxis='coloraxis1', y=a_a_range, x=a_b_range), row=1, col=1)
fig.add_trace(go.Heatmap(z=geometric_losses, zmin=cmin_row1, zmax=cmax_row1, coloraxis='coloraxis1', y=a_a_range, x=a_b_range), row=1, col=2)
fig.add_trace(go.Heatmap(z=amplitude_losses, zmin=cmin_row1, zmax=cmax_row1, coloraxis='coloraxis1', y=a_a_range, x=a_b_range), row=1, col=3)

fig.add_trace(go.Heatmap(z=np.gradient(euclidean_losses)[0], zmin=cmin_row2, zmax=cmax_row2, coloraxis='coloraxis2', y=a_a_range, x=a_b_range), row=2, col=1)
fig.add_trace(go.Heatmap(z=np.gradient(geometric_losses)[0], zmin=cmin_row2, zmax=cmax_row2, coloraxis='coloraxis2', y=a_a_range, x=a_b_range), row=2, col=2)
fig.add_trace(go.Heatmap(z=np.gradient(amplitude_losses)[0], zmin=cmin_row2, zmax=cmax_row2, coloraxis='coloraxis2', y=a_a_range, x=a_b_range), row=2, col=3)

fig.add_trace(go.Heatmap(z=np.gradient(euclidean_losses)[1], zmin=cmin_row3, zmax=cmax_row3, coloraxis='coloraxis3', y=a_a_range, x=a_b_range), row=3, col=1)
fig.add_trace(go.Heatmap(z=np.gradient(geometric_losses)[1], zmin=cmin_row3, zmax=cmax_row3, coloraxis='coloraxis3', y=a_a_range, x=a_b_range), row=3, col=2)
fig.add_trace(go.Heatmap(z=np.gradient(amplitude_losses)[1], zmin=cmin_row3, zmax=cmax_row3, coloraxis='coloraxis3', y=a_a_range, x=a_b_range), row=3, col=3)

fig.add_trace(go.Heatmap(z=np.abs(euclidean_fft), zmin=0, zmax=0.005, coloraxis='coloraxis4', y=a_a_range, x=a_b_range), row=4, col=1)
fig.add_trace(go.Heatmap(z=np.abs(geometric_fft), zmin=0, zmax=0.005, coloraxis='coloraxis4', y=a_a_range, x=a_b_range), row=4, col=2)
fig.add_trace(go.Heatmap(z=np.abs(amplitude_fft), zmin=0, zmax=0.005, coloraxis='coloraxis4', y=a_a_range, x=a_b_range), row=4, col=3)

[fig.add_trace(go.Scatter(x=[a0_b], y=[a0_a], mode='markers', marker=dict(color='red', size=10, symbol='x'), showlegend=False), row=x, col=y) for x in range(1,4) for y in range(1,4)]

fig.update_layout(
    height=1200, width=900,
    title_text="Loss Analysis (Correct Relative Amplitude)",

    # Define the first color axis for Row 1
    coloraxis1=dict(
        cmin=cmin_row1,
        cmax=cmax_row1,
        colorbar=dict(
            title="Loss",
            thickness=15,
            x=1.02, # Position to the right of the plot
            y=1.04, # Vertical position (top row)
            yanchor='top',
            len=0.25   # Length of the colorbar
        )
    ),
    # Define the second color axis for Row 2
    coloraxis2=dict(
        colorscale='RdBu',
        cmin=cmin_row2,
        cmax=cmax_row2,
        colorbar=dict(
            title="Grad X",
            thickness=15,
            x=1.02,
            y=0.645, # Vertical position (middle row)
            yanchor='middle',
            len=0.25
        )
    ),
    # Define the third color axis for Row 3
    coloraxis3=dict(
        colorscale='RdBu',
        cmin=cmin_row3,
        cmax=cmax_row3,
        colorbar=dict(
            title="Grad Y",
            thickness=15,
            x=1.02,
            y=0.25, # Vertical position (bottom row)
            yanchor='bottom',
            len=0.25
        )
    ),
    # Define the fourth color axis for Row 4
    coloraxis4=dict(
        cmin=cmin_row4,
        cmax=cmax_row4,
        colorbar=dict(
            title="FFT Magnitude",
            thickness=15,
            x=1.02,
            y=-0.02, # Vertical position (bottom row)
            yanchor='bottom',
            len=0.25
        )
    ),
)


fig.show()

In [12]:
# Initialize the figure
fig = make_subplots(
    rows=4, cols=3,
    column_titles=['Euclidean Loss', 'Geometric Loss', 'Amplitude Loss'],
    row_titles=['Loss Field', 'Gradient x', 'Gradient y', 'FFT Magnitude (log2)'],
    shared_xaxes=True, shared_yaxes=True
)

# --- Data Preparation ---
# Calculate gradients for each loss
grad_euclidean = np.sign(np.gradient(euclidean_losses))
grad_geometric = np.sign(np.gradient(geometric_losses))
grad_amplitude = np.sign(np.gradient(amplitude_losses))

# Calculate FFTs for each loss
euclidean_fft = np.fft.fftshift(np.fft.fft2(euclidean_losses, norm='ortho'))
geometric_fft = np.fft.fftshift(np.fft.fft2(geometric_losses, norm='ortho'))
amplitude_fft = np.fft.fftshift(np.fft.fft2(amplitude_losses, norm='ortho'))


# --- Row 1: Loss Field ---
# Each trace will be normalized independently based on its own min/max values.
fig.add_trace(go.Heatmap(z=euclidean_losses, y=a_a_range, x=a_b_range, zmin=0), row=1, col=1)
fig.add_trace(go.Heatmap(z=geometric_losses, y=a_a_range, x=a_b_range, zmin=0), row=1, col=2)
fig.add_trace(go.Heatmap(z=amplitude_losses, y=a_a_range, x=a_b_range, zmin=0), row=1, col=3)

# --- Row 2: Gradient x ---
# A diverging colorscale is used for gradients.
fig.add_trace(go.Heatmap(z=grad_euclidean[0], colorscale='RdBu', y=a_a_range, x=a_b_range, zmin=-np.max(np.abs(grad_euclidean[0])), zmax=np.max(np.abs(grad_euclidean[0]))), row=2, col=1)
fig.add_trace(go.Heatmap(z=grad_geometric[0], colorscale='RdBu', y=a_a_range, x=a_b_range, zmin=-np.max(np.abs(grad_geometric[0])), zmax=np.max(np.abs(grad_geometric[0]))), row=2, col=2)
fig.add_trace(go.Heatmap(z=grad_amplitude[0], colorscale='RdBu', y=a_a_range, x=a_b_range, zmin=-np.max(np.abs(grad_amplitude[0])), zmax=np.max(np.abs(grad_amplitude[0]))), row=2, col=3)

# --- Row 3: Gradient y ---
fig.add_trace(go.Heatmap(z=grad_euclidean[1], colorscale='RdBu', y=a_a_range, x=a_b_range, zmin=-np.max(np.abs(grad_euclidean[1])), zmax=np.max(np.abs(grad_euclidean[1]))), row=3, col=1)
fig.add_trace(go.Heatmap(z=grad_geometric[1], colorscale='RdBu', y=a_a_range, x=a_b_range, zmin=-np.max(np.abs(grad_geometric[1])), zmax=np.max(np.abs(grad_geometric[1]))), row=3, col=2)
fig.add_trace(go.Heatmap(z=grad_amplitude[1], colorscale='RdBu', y=a_a_range, x=a_b_range, zmin=-np.max(np.abs(grad_amplitude[1])), zmax=np.max(np.abs(grad_amplitude[1]))), row=3, col=3)

# --- Row 4: FFT Magnitude ---
fig.add_trace(go.Heatmap(z=np.log2(np.abs(euclidean_fft)), y=a_a_range, x=a_b_range, zmin=0), row=4, col=1)
fig.add_trace(go.Heatmap(z=np.log2(np.abs(geometric_fft)), y=a_a_range, x=a_b_range, zmin=0), row=4, col=2)
fig.add_trace(go.Heatmap(z=np.log2(np.abs(amplitude_fft)), y=a_a_range, x=a_b_range, zmin=0), row=4, col=3)

# --- Add Markers ---
[fig.add_trace(go.Scatter(x=[a0_b], y=[a0_a], mode='markers', marker=dict(color='red', size=10, symbol='x'), showlegend=False), row=x, col=y) for x in range(1, 4) for y in range(1, 4)]

# --- Update Layout ---
# The shared coloraxis definitions are no longer needed.
fig.update_layout(
    height=1200, width=900,
    title_text="Loss Analysis with Independent Min-Max Normalization"
)

fig.show()

/tmp/ipykernel_50705/2784687459.py:39: RuntimeWarning: divide by zero encountered in log2
  fig.add_trace(go.Heatmap(z=np.log2(np.abs(euclidean_fft)), y=a_a_range, x=a_b_range, zmin=0), row=4, col=1)
/tmp/ipykernel_50705/2784687459.py:40: RuntimeWarning: divide by zero encountered in log2
  fig.add_trace(go.Heatmap(z=np.log2(np.abs(geometric_fft)), y=a_a_range, x=a_b_range, zmin=0), row=4, col=2)
/tmp/ipykernel_50705/2784687459.py:41: RuntimeWarning: divide by zero encountered in log2
  fig.add_trace(go.Heatmap(z=np.log2(np.abs(amplitude_fft)), y=a_a_range, x=a_b_range, zmin=0), row=4, col=3)


In [13]:
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'is_3d': True}, {'is_3d': True}, {'is_3d': True}]],
    column_titles=['Euclidean Loss', 'Geometric Loss', 'Amplitude Loss'],
    row_titles=['Loss Field'],
    shared_xaxes=True, shared_yaxes=True
)

fig.add_trace(go.Surface(z=euclidean_losses, y=a_a_range, x=a_b_range), row=1, col=1)
fig.add_trace(go.Surface(z=geometric_losses,y=a_a_range, x=a_b_range), row=1, col=2)
fig.add_trace(go.Surface(z=amplitude_losses,y=a_a_range, x=a_b_range), row=1, col=3)
fig.add_trace(go.Scatter3d(x=[a0_b], y=[a0_a], z=[0], mode='markers', marker=dict(color='red', size=5, symbol='x'), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter3d(x=[a0_b], y=[a0_a], z=[0], mode='markers', marker=dict(color='red', size=5, symbol='x'), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter3d(x=[a0_b], y=[a0_a], z=[0], mode='markers', marker=dict(color='red', size=5, symbol='x'), showlegend=False), row=1, col=3)   

fig.update_layout(title=dict(text='Loss Surface'), autosize=True,
                  width=2400, height=800,
                  margin=dict(l=65, r=50, b=65, t=90))

fig.show()